# Training Results Viewer

This notebook allows you to load and visualize the results of training an autoencoder on the EEG dataset. You can specify which fold's results to load and then view the reconstruction performance of the model on the test set.

Requirements:
 - Model checkpoints at path: core.settings.FINETUNE_OUTPUT_DIR/your_model_name
 - Preprocessed data at path: core.settings.FINETUNE_DIR

*To obtain this data, please run make_dataset.py and run_general_finetuning.py beforehand*
## Specify parameters

In [1]:

import json

# specify which fold to load (0-5)
fold_to_load = 3
model_name = "wavelet_1"
target_mode = "wavelet" # options: "wavelet", "superlet", "autoencoder"
epochs_to_load = []
load_best = True

## Load model and data

In [2]:
from core.architectures import EEGAutoencoder, EEGSpectralDecoder
from core.datasets import KFoldSplitter
import core.settings as settings
import core.utils as core_utils
from pathlib import Path
import matplotlib.pyplot as plt
import os
from libs.labram import utils
from einops import rearrange
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
import matplotlib.ticker as ticker


if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if target_mode not in ["autoencoder", "wavelet", "superlet"]:
    raise ValueError("target_mode must be 'autoencoder' 'wavelet' or 'superlet'")

splitter = KFoldSplitter(data_dir=settings.FINETUNE_EPOCHS_DIR, seed=settings.SEED, k_folds=6)
folds = splitter.split_by_subject()


fold = folds[fold_to_load]

train_dataset, val_dataset, test_dataset = fold.initialize_datasets(target_mode=target_mode,
window_size=settings.WINDOW_SIZE, wavelet_dir=settings.FINETUNE_WAVELET_DIR, superlet_dir=settings.FINETUNE_SUPERLET_DIR)

ch_names = settings.CHORDER_STANDARD
# cut off "-REF" and "EEG " from channel names for compatibility with get_input_chans function
ch_names = [name.split(' ')[-1].split('-')[0] for name in ch_names]
input_chans = utils.get_input_chans(ch_names)


fold_path = settings.FINETUNE_OUTPUT_DIR / model_name / Path(str(fold_to_load + 1))
metadata_path = settings.FINETUNE_OUTPUT_DIR / model_name

checkpoint_paths = [fold_path / f"checkpoint-{epoch}.pth" for epoch in epochs_to_load]

if load_best:
    checkpoint_paths.append(fold_path / "checkpoint-best_test.pth")

with open(metadata_path / 'metadata.json', 'r') as f:
    metadata = json.load(f)

params = metadata["hyperparameters"]

model = None
if target_mode == "autoencoder":
    model = EEGAutoencoder(**settings.AUTOENCODER_PARAMS).eval()
if target_mode in ["wavelet", "superlet"]:
    model = EEGSpectralDecoder(**settings.SPECTRAL_DECODER_PARAMS).eval()

## Visualize reconstructions

In [3]:
slider = widgets.IntSlider(
        min=0,
        max=len(test_dataset)-1,
        step=1,
        value=0,
        description=f"Sample: ",
        layout = widgets.Layout(width="100%", align="center"),
    )

save_button = widgets.Button(
    description="Save Figure",
    layout=widgets.Layout(width="100%", align="center"),
)
save_message = widgets.Output()

def set_formatting(ax, target_mode, im=None):
    if target_mode == "autoencoder":
        ax.set_xlabel("Time (ms)")
        ax.set_ylabel("Amplitude [uV]")
        ax.legend()
    elif target_mode in ["wavelet", "superlet"]:
        ax.set_xlabel("Time (ms)")
        ax.set_ylabel("Frequency (Hz)")
        ax.set_yscale("log") # because of geomspace
        plt.colorbar(im, label='Power', ax=ax)
        my_ticks = [4, 5, 10, 20, 50]
        ax.set_yticks(my_ticks)
        ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
        ax.yaxis.set_minor_formatter(ticker.NullFormatter())
        ax.yaxis.set_minor_locator(ticker.NullLocator())

def show_reconstructions(sample_idx, save= True):
    fig = plt.figure(figsize=(16, 6*len(checkpoint_paths)))
    param_text = f"LR: {params['lr']} | WD: {params['weight_decay']}"
    plt.suptitle(f"{target_mode} training results for model {model_name}\n{param_text}", y=1.05, fontweight="bold")

    gs = fig.add_gridspec(nrows=len(checkpoint_paths),ncols=1)

    epoch = test_dataset[sample_idx][0]
    test_input = epoch.unsqueeze(0)
    device = next(model.parameters()).device
    test_input = rearrange(test_input, 'B N (A T) -> B N A T', T=200)
    test_input = test_input.float().to(device, non_blocking=True) / 100

    target = test_dataset[sample_idx][1]
    channel_true = None
    if target_mode == "autoencoder":
        channel_true = target[0, :].cpu().numpy()
    elif target_mode in ["wavelet", "superlet"]:
        channel_true = target[0, :, :].cpu().numpy()
    #true_wave_avg = np.mean(epoch.cpu().numpy(), axis=0)

    for i, checkpoint_path in enumerate(checkpoint_paths):
        try:
            if not checkpoint_path.exists():
                print(f"No checkpoint found at {checkpoint_path}")
                continue
            recon_params = settings.AUTOENCODER_PARAMS if target_mode == "autoencoder" else settings.SPECTRAL_DECODER_PARAMS
            reconstruction, epoch_idx = core_utils.reconstruct_model_output(checkpoint_path= checkpoint_path, model=model, input=test_input, input_chans=input_chans, window_size=settings.WINDOW_SIZE, **recon_params)

            # extract the reconstructed waveform for the first channel
            channel_recon = reconstruction[0, :].cpu().numpy()

            if epoch_idx == "best_test":
                epoch_idx = str(metadata['results']['max_r2_test_epochs'][fold_to_load]) + ' (Best test results)'

            r2 = core_utils.calculate_r2(channel_true, channel_recon)
            time = np.arange(channel_true.shape[-1]) / settings.RSFREQ * 1000

            subfig = fig.add_subfigure(gs[i])
            if target_mode == "autoencoder":

                ax1 = subfig.add_subplot(len(checkpoint_paths), 1, i + 1)
                ax1.plot(time, channel_true, label='True', alpha=0.7)
                ax1.plot(time, channel_recon, label='Reconstructed', linestyle='--', alpha=0.9)
                ax1.set_title(f"Fold {fold_to_load}, test sample {sample_idx} - Epoch: {epoch_idx}", loc='left')
                ax1.text(0.95, 0.05, f"R²: {r2:.4f}", transform=ax1.transAxes, ha='right', va='top', fontsize=10,
                         bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

                set_formatting(ax1, target_mode)

            elif target_mode == "wavelet":
                subfig.suptitle(f"Fold {fold_to_load}, test sample {sample_idx} - Epoch: {epoch_idx}")
                v_max = max(channel_true.max(), channel_recon.max())
                freq_window = settings.SPECTRAL_DECODER_PARAMS["freq_window"]
                n_freqs = settings.SPECTRAL_DECODER_PARAMS["n_freqs"]
                freqs = np.geomspace(freq_window[0], freq_window[1], n_freqs)
                ax1 = subfig.add_subplot(len(checkpoint_paths), 3, 3 * i + 1)
                im1=ax1.pcolormesh(time, freqs, channel_true, vmin=0, vmax=v_max)
                ax1.set_title("True", loc='center')
                set_formatting(ax1, target_mode, im1)

                ax2 = subfig.add_subplot(len(checkpoint_paths), 3, 3 * i + 2)
                im2=ax2.pcolormesh(time, freqs, channel_recon, shading="gouraud", vmin=0, vmax=v_max)
                ax2.set_title("Reconstructed",
                              loc='center')
                set_formatting(ax2, target_mode, im2)

                ax3 = subfig.add_subplot(len(checkpoint_paths), 3, 3 * i + 3)
                im3=ax3.pcolormesh(time, freqs, np.abs(channel_true - channel_recon), shading="gouraud", vmin=0, vmax=v_max)
                ax3.set_title("Absolute Error",
                              loc='center')
                set_formatting(ax3, target_mode, im3)
                plt.text(0.95, 0.05, f"R²: {r2:.4f}", transform=ax3.transAxes, ha='right', va='top', fontsize=10,
                         bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
        except Exception as e:
            print(e)

    plt.tight_layout()
    if save:
        save_dir = settings.PLOT_DIR / model_name / "reconstructions"
        save_dir.mkdir(parents=True, exist_ok=True)
        filename = save_dir / f"sample_{sample_idx}_reconstruction.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print("Saved figure to:", filename)
    plt.show()


save_button.on_click(show_reconstructions)


interact(
    show_reconstructions,
    sample_idx= slider,
    save=widgets.fixed(False)
)

display(save_button, save_message)
def on_save_clicked(b):
    show_reconstructions(slider.value, save=True)

save_button.on_click(on_save_clicked)

interactive(children=(IntSlider(value=0, description='Sample: ', layout=Layout(width='100%'), max=537), Output…

Button(description='Save Figure', layout=Layout(width='100%'), style=ButtonStyle())

Output()